# 第二次作業｜團體出遊分帳助手

**學校：** 國立政治大學  
**學號：** 112303086  
**系級：** 會計四  
**姓名：** 曾尉庭  
**選題：** 題 1，自行出題

## 重點說明

這次想做一個出遊分帳程式，把每個人先付的錢填進去，就能算出還要付多少、要轉給誰。

### 自己設定的功能

1. 輸入姓名和代付金額，人數可以自己改。
2. 算出總支出和每個人要分攤的金額。
3. 判斷誰要收錢、誰要補錢，再列出轉帳清單。
4. 分不整除的金額也能分完，不會少一分。
5. 名單或金額填錯時，顯示提醒。

### 使用的本週功能

| 本週功能 | 用在哪裡 |
|---|---|
| `for` 迴圈 | 檢查資料、逐人列出結果 |
| `if / elif / else` | 判斷應收、應付或已結清 |
| `while` 迴圈 | 配對付款和收款的人，直到結清 |
| 列表生成式 | 換算金額、算差額、整理名單 |
| `def`、`return` | 把分帳寫成函式，回傳結果 |
| `f-string` | 放入姓名和金額，顯示到小數點後兩位 |

## 1. 分帳程式

這邊是大家平均分攤，金額填整數元，姓名跟金額的順序要一樣。計算時先換成分，除不盡的部分照名單順序多分攤一分；已付金額減掉分攤金額，就是要收回或補上的錢。

**AI 協助：** 題目、分帳規則和下方函式。

In [1]:
# AI 協助：ChatGPT（OpenAI）協助設計分帳、配對轉帳與整理輸出。
def split_bill(names, paid):
    # 先確認人數、姓名和金額，錯誤時直接回傳提醒
    if len(names) == 0 or len(names) != len(paid):
        return '輸入錯誤：姓名和金額要一一對應，而且至少要有一個人。'
    for i in range(len(names)):
        if type(names[i]) != str or names[i].strip() == '' or names[i] != names[i].strip():
            return '輸入錯誤：姓名不能空白，前後也不要留空格。'
        if names[i] in names[:i]:
            return '輸入錯誤：姓名不能重複，可以加編號來區分。'
        if type(paid[i]) != int or paid[i] < 0:
            return '輸入錯誤：代付金額請填 0 或正整數，單位是元。'

    # 1. 把元換成分，再算每個人要分攤多少
    paid_cents = [amount * 100 for amount in paid]
    total = sum(paid_cents)
    share = total // len(names)
    shares = [share for name in names]
    # 除不盡的分，依名單順序各多分攤一分
    for i in range(total % len(names)):
        shares[i] += 1
    balances = [paid_cents[i] - shares[i] for i in range(len(names))]

    # 2. 逐人列出代付、分攤和應收應付
    report = f'總支出：{total / 100:.2f} 元｜共 {len(names)} 人\n\n每人分帳結果：\n'
    for i in range(len(names)):
        if balances[i] > 0:
            status = f'應收 {balances[i] / 100:.2f} 元'
        elif balances[i] < 0:
            status = f'應付 {-balances[i] / 100:.2f} 元'
        else:
            status = '已結清'
        report += f'{names[i]}：代付 {paid_cents[i] / 100:.2f} 元｜分攤 {shares[i] / 100:.2f} 元｜{status}\n'

    # 3. 分成應付款和應收款兩份名單，逐筆配對
    debtors = [i for i in range(len(names)) if balances[i] < 0]
    creditors = [i for i in range(len(names)) if balances[i] > 0]
    remaining = balances.copy()
    transfers = ''
    d = 0
    c = 0
    while d < len(debtors) and c < len(creditors):
        sender = debtors[d]
        receiver = creditors[c]
        amount = min(-remaining[sender], remaining[receiver])
        transfers += f'{names[sender]} → {names[receiver]}：{amount / 100:.2f} 元\n'
        remaining[sender] += amount
        remaining[receiver] -= amount
        if remaining[sender] == 0:
            d += 1
        if remaining[receiver] == 0:
            c += 1

    if transfers == '':
        transfers = '不用轉帳，大家都已經結清。\n'

    return f'{report}\n建議轉帳順序：\n{transfers}'


## 2. 四個人出遊的例子

小安先付 1,200 元、小庭付 600 元，另外兩人還沒付，所以每個人要分攤 450 元。改下面的姓名和金額，就可以換一組資料。

**AI 協助：** 範例資料和結果核對。

In [2]:
names = ['小安', '小庭', '小宇', '小晴']
paid = [1200, 600, 0, 0]
print(split_bill(names, paid))

總支出：1800.00 元｜共 4 人

每人分帳結果：
小安：代付 1200.00 元｜分攤 450.00 元｜應收 750.00 元
小庭：代付 600.00 元｜分攤 450.00 元｜應收 150.00 元
小宇：代付 0.00 元｜分攤 450.00 元｜應付 450.00 元
小晴：代付 0.00 元｜分攤 450.00 元｜應付 450.00 元

建議轉帳順序：
小宇 → 小安：450.00 元
小晴 → 小安：300.00 元
小晴 → 小庭：150.00 元



## 3. 分不整除的情況

三個人分 100 元，如果都算 33.33 元就會少一分，所以第一位算 33.34 元，另外兩位算 33.33 元。

**AI 協助：** 尾差範例和金額核對。

In [3]:
print(split_bill(['小安', '小庭', '小宇'], [100, 0, 0]))

總支出：100.00 元｜共 3 人

每人分帳結果：
小安：代付 100.00 元｜分攤 33.34 元｜應收 66.66 元
小庭：代付 0.00 元｜分攤 33.33 元｜應付 33.33 元
小宇：代付 0.00 元｜分攤 33.33 元｜應付 33.33 元

建議轉帳順序：
小庭 → 小安：33.33 元
小宇 → 小安：33.33 元



## 4. 不用轉帳或填錯資料

大家付一樣多就不用轉帳，全部填 0 也可以。下面順便試一下空名單、人數不一致、負數、小數和重複姓名。

**AI 協助：** 檢查情境和錯誤提醒。

In [4]:
print('【每個人都付一樣多】')
print(split_bill(['小安', '小庭', '小宇'], [300, 300, 300]))
print('【全部都是 0 元】')
print(split_bill(['小安', '小庭', '小宇'], [0, 0, 0]))
print('【空名單】')
print(split_bill([], []))
print('【姓名和金額數量不同】')
print(split_bill(['小安', '小庭'], [100]))
print('【負數】')
print(split_bill(['小安', '小庭'], [100, -20]))
print('【非整數金額】')
print(split_bill(['小安', '小庭'], [100, 20.5]))
print('【重複姓名】')
print(split_bill(['小安', '小安'], [100, 0]))

【每個人都付一樣多】
總支出：900.00 元｜共 3 人

每人分帳結果：
小安：代付 300.00 元｜分攤 300.00 元｜已結清
小庭：代付 300.00 元｜分攤 300.00 元｜已結清
小宇：代付 300.00 元｜分攤 300.00 元｜已結清

建議轉帳順序：
不用轉帳，大家都已經結清。

【全部都是 0 元】
總支出：0.00 元｜共 3 人

每人分帳結果：
小安：代付 0.00 元｜分攤 0.00 元｜已結清
小庭：代付 0.00 元｜分攤 0.00 元｜已結清
小宇：代付 0.00 元｜分攤 0.00 元｜已結清

建議轉帳順序：
不用轉帳，大家都已經結清。

【空名單】
輸入錯誤：姓名和金額要一一對應，而且至少要有一個人。
【姓名和金額數量不同】
輸入錯誤：姓名和金額要一一對應，而且至少要有一個人。
【負數】
輸入錯誤：代付金額請填 0 或正整數，單位是元。
【非整數金額】
輸入錯誤：代付金額請填 0 或正整數，單位是元。
【重複姓名】
輸入錯誤：姓名不能重複，可以加編號來區分。


## 5. 核對結果

這邊用 `assert` 檢查算出來的結果，有錯就會停下來。這是額外檢查，不算在本週六種功能裡。

**AI 協助：** 檢查條件和測試。

In [5]:
result = split_bill(['小安', '小庭', '小宇', '小晴'], [1200, 600, 0, 0])
assert '小宇 → 小安：450.00 元' in result
assert '小晴 → 小安：300.00 元' in result
assert '小晴 → 小庭：150.00 元' in result
result = split_bill(['小安', '小庭', '小宇'], [100, 0, 0])
assert '分攤 33.34 元｜應收 66.66 元' in result
assert '小庭 → 小安：33.33 元' in result
assert '小宇 → 小安：33.33 元' in result
assert '不用轉帳' in split_bill(['小安', '小庭'], [0, 0])
assert split_bill([], []).startswith('輸入錯誤：')
assert split_bill(['小安', '小庭'], [100]).startswith('輸入錯誤：')
assert split_bill(['小安', '小庭'], [100, -20]).startswith('輸入錯誤：')
assert split_bill(['小安', '小庭'], [100, 20.5]).startswith('輸入錯誤：')
assert split_bill(['小安', '小安'], [100, 0]).startswith('輸入錯誤：')
print('檢查通過：一般分帳、除不盡、零元、人數不一致、負數、非整數及重複姓名。')

檢查通過：一般分帳、除不盡、零元、人數不一致、負數、非整數及重複姓名。


## 補充

這個程式只適合大家平均分攤，轉帳順序照名單排，沒有保證筆數最少。

這份作業有用 ChatGPT 協助題目、程式、測試和文字，各段的功能都有註明。